In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    to_date,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_underlaw_rpt(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema=(
        #      "UnderLawId string,"
        #      "SourceSystem string,"
        #      "PortfolioKey string,"
        #      "PortfolioName string,"
        #      "IngestionTimestamp timestamp,"
        #      "BatchId string,"
        #      "UnderLawKey string,"
        #      "CDI string,"
        #      "TaxRegistrationNumber string,"
        #      "ApplicationNumber string,"
        #      "ApplicationStatusCode string,"
        #      "ApplicationStatus string,"
        #      "ApplicationStatusDate timestamp,"
        #      "TypeOfCourt string,"
        #      "CourtDate date,"
        #      "RecordDate date,"
        #      "SubmissionDate date,"
        #      "InitiationDate date,"
        #      "CourtOutcomeCode string,"
        #      "CourtOutcome string,"
        #      "CompletionReasonCode int,"
        #      "CompletionReason string,"
        #      "LawSystem string,"
        #      "PlatformName string,"
        #      "InLawFlag int,"
        #      "LawStatusCode string,"
        #      "LawStatus string,"
        #      "CalcLawStatusCode string,"
        #      "CalcLawStatus string,"
        #      "UnderLawType string,"
        #      "UnderLawTypeCode string,"
        #      "TemporaryCourtDecisionDate timestamp,"
        #      "DeductionDate timestamp,"
        #      "CountyCourtCode string,"
        #      "CountyCourt string,"
        #      "InterbankStatus3869Code string,"
        #      "InterbankStatus3869 string,"
        #      "SubmissionDateGroup string,"
        #      "ResubmissionStatusFlag string,"
        #      "ResubmissionStatusCode string, "
        #      "ResubmissionStage string, "
        #      "AppealCourtDate timestamp, "
        #      "CepalEligibilityDate timestamp, "
        #      "BusinessDate date"
        #    )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.UnderLaw"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "rpt",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== rpt_applications ==================

        applications_extra_filters = None

        if applications_extra_filters is not None:
            applications_extra_filters.append(date_filter)
            applications_final_filters = " AND ".join(applications_extra_filters)
        else:
            applications_final_filters = date_filter

        applications_selected_fields = [
            "ApplicantVatNumber",
            "ApplicationNumber",
            "Status",
            "StatusLiteral",
            "StatusStartDate",
            "InitiationDate",
            "CompletionReasonLiteral",
            "CompletionReason",
            "LawSystemName",
            "PlatformName",
            "Law",
            "SubmisionDate",
            "BusinessDate",
        ]

        applications = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "dwpltfrm_Stgapplications",
            applications_final_filters,
            applications_selected_fields,
        )

        # Guide §5: this notebook now owns ONLY Bridge1/Bridge2/L4605.
        # The L4738 / N4738 family moved to underlaw_cepalbiplatform.
        applications = applications.filter("PlatformName IN ('Bridge1','Bridge2','L4605')")

        # A data frame that included all the other laws    EXCEPT    N4738/2020/Εξωδικαστικός μηχανισμός.
        applications_except_N4738 = applications.filter(
            "LawSystemName != 'Εξωδικαστικός μηχανισμός'"
        ).distinct()
        

       

        applications_final_df = applications_except_N4738

        

        # ================= rpt_application_subsidy ==========

        application_subsidy_extra_filters = None

        if application_subsidy_extra_filters is not None:
            application_subsidy_extra_filters.append(date_filter)
            application_subsidy_final_filters = " AND ".join(
                application_subsidy_extra_filters
            )
        else:
            application_subsidy_final_filters = date_filter

        application_subsidy_selected_fields = [
            "ApplicationNumber",
            "Law",
            "CDI",
            "cepal_setEligibiltyDate",
        ]

        application_subsidy = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "dwpltfrm_Stgapplicationsubsidy",
            application_subsidy_final_filters,
            application_subsidy_selected_fields,
        )

        

        # ================ join ===========

        underlaw_df = applications_final_df.join(
            application_subsidy,
            (applications_final_df.Law == application_subsidy.Law)
            & (
                applications_final_df.ApplicationNumber
                == application_subsidy.ApplicationNumber
            ),
            "left",
        )

        # ================  ================

        UnderLawId_primary_key_fields = [
            applications_final_df.ApplicationNumber,
            applications_final_df.Law,
            applications_final_df.LawSystemName,
            applications_final_df.ApplicantVatNumber,
            coalesce(application_subsidy.CDI, lit("-1")),
            lit("rpt"),
        ]

        UnderLawKey_primary_key_fields = [
            applications_final_df.ApplicationNumber,
            applications_final_df.Law,
            applications_final_df.LawSystemName,
            applications_final_df.ApplicantVatNumber,
            coalesce(application_subsidy.CDI, lit("-1")),
        ]

        underlaw_df = (
            underlaw_df.withColumn(
                "UnderLawId",
                sha2(
                    concat_ws(";", *UnderLawId_primary_key_fields),
                    256,
                ),
            )
            .withColumn("UnderLawKey", concat_ws(";", *UnderLawKey_primary_key_fields))
            .withColumn(
                "SubmissionDate",
                when(
                    to_date(underlaw_df.SubmisionDate, "yyyy-MM-dd")
                    > lit("1900-01-01"),
                    to_date(underlaw_df.SubmisionDate, "yyyy-MM-dd"),
                ).otherwise(lit(None)),
            )
        )


        # ================== Selection of Final Columns ==================

        underlaw_rpt = underlaw_df.select(
            underlaw_df.UnderLawId.alias("UnderLawId"),
            lit("rpt").alias("SourceSystem"),
            lit(None).cast("string").alias("PortfolioKey"),
            lit(None).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            underlaw_df.UnderLawKey.alias("UnderLawKey"),
            application_subsidy.CDI.alias("CDI"),
            underlaw_df.ApplicantVatNumber.alias("TaxRegistrationNumber"),
            applications_final_df.ApplicationNumber.alias("ApplicationNumber"),
            underlaw_df.Status.cast("string").alias("ApplicationStatusCode"),
            underlaw_df.StatusLiteral.cast("string").alias("ApplicationStatus"),
            underlaw_df.StatusStartDate.cast("timestamp").alias(
                "ApplicationStatusDate"
            ),
            lit(None).cast("string").alias("TypeOfCourt"),
            lit(None).cast("date").alias("CourtDate"),
            lit(None).alias("RecordDate"),
            underlaw_df.SubmissionDate.alias("SubmissionDate"),
            underlaw_df.InitiationDate.cast("date").alias("InitiationDate"),
            lit(None).cast("string").alias("CourtOutcomeCode"),
            lit(None).cast("string").alias("CourtOutcome"),
            underlaw_df.CompletionReason.alias("CompletionReasonCode"),
            underlaw_df.CompletionReasonLiteral.alias("CompletionReason"),
            underlaw_df.LawSystemName.alias("LawSystem"),
            underlaw_df.PlatformName.alias("PlatformName"),
            lit(None).cast("int").alias("InLawFlag"),
            lit(None).cast("string").alias("LawStatusCode"),
            lit(None).cast("string").alias("LawStatus"),
            lit(None).alias("CalcLawStatusCode"),
            lit(None).alias("CalcLawStatus"),
            applications_final_df.Law.alias("UnderLawType"),
            applications_final_df.Law.alias("UnderLawTypeCode"),
            lit(None).alias("TemporaryCourtDecisionDate"),
            lit(None).alias("DeductionDate"),
            lit(None).alias("CountyCourtCode"),
            lit(None).alias("CountyCourt"),
            lit(None).alias("InterbankStatus3869Code"),
            lit(None).alias("InterbankStatus3869"),
            lit(None).alias("SubmissionDateGroup"),
            lit(None).alias("ResubmissionStatusFlag"),
            lit(None).alias("ResubmissionStatusCode"),
            lit(None).alias("ResubmissionStage"),
            lit(None).alias("AppealCourtDate"),
            application_subsidy.cepal_setEligibiltyDate.alias("CepalEligibilityDate"),
            applications_final_df.BusinessDate.alias("BusinessDate"),
        )
        underlaw_rpt = underlaw_rpt.distinct()

        # ========================== Append ==========================
        underlaw_rpt.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

        logging.info("We did it!")


In [0]:
load_curated_underlaw_rpt(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)